# Code for analysis of crispresso editing information

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [2]:
def crispresso_compiler(samp, sample_id, folder_path):
    """ 
    Takes in list of sample names (see cell above)
    Returns compiled dictionary of dataframes containing editing information for each sensor
    
    Includes:
    1. corr_perc = pure correct editing
    2. target_base_edit_perc = target base editing perc (including edits with bystander editing)
    3. wt_perc
    4. byproduct information (indels, substitutions, ambiguous)
    """

    df_edits = []
    for k in samp: 
        
        try:
            concated = pd.read_csv(f'{folder_path}/{k}_crispresso_aggregated.csv')

        except:
            concated = pd.read_csv(f'{folder_path}/{k}_crispresso_aggregated_v2.csv')
            
        concated = concated.fillna(0)

        #somehow wasn't able to find the initial set of code that I used for this...

        #go through all of the samples and do it step by step
        sample_ids = []
        #sample_num = []
        rii = []
        raaa = []
        r_lowqual = []
        ra_hdr = []
        ra_wt = []
        r_unaligned = []
        no_edit = []
        correct = []
        target_base_editing = []
        byproduct_all = []
        byproduct_indel = []
        byproduct_sub = []
        byproduct_ambig = []
        for i in np.unique(concated['Guide_ID']):
            sample_ids.append(i)

            subset = concated[concated['Guide_ID']==i]

            wt = subset[subset['Amplicon']=='Reference']
            edit = subset[subset['Amplicon']=='HDR']

            #sample_num.append(wt['sample_num'].values[0])
            rii1 = wt['Reads_in_input'].values[0]
            r1 = wt['Reads_aligned_all_amplicons'].values[0]
            rii.append(rii1)
            raaa.append(r1)
            r_lowqual.append(rii1-r1)

            r2 = edit['Reads_aligned'].values[0]
            r3 = wt['Reads_aligned'].values[0]
            ra_hdr.append(r2)
            ra_wt.append(r3)
            r_unaligned.append(r1 - (r2+r3))

            no_edit.append(wt['Unmodified'].values[0])
            correct.append(edit['Unmodified'].values[0])
            target_base_editing.append(edit['Modified'].values[0] + edit['Unmodified'].values[0])

            byprod_all = wt['Modified'].values[0] + edit['Modified'].values[0] + (r1 - (r2+r3)) #add unaligned reads
            sub_all = wt['Only Substitutions'].values[0] + edit['Only Substitutions'].values[0]
            indel_all = wt['Only Deletions'].values[0] + wt['Only Insertions'].values[0] + wt['Insertions and Deletions'].values[0] + edit['Only Deletions'].values[0] + edit['Only Insertions'].values[0] + edit['Insertions and Deletions'].values[0]
            ambig_all = byprod_all - sub_all - indel_all

            byproduct_all.append(byprod_all)
            byproduct_indel.append(indel_all)
            byproduct_sub.append(sub_all)
            byproduct_ambig.append(ambig_all)


        cols = ["Guide_ID", "Reads_in_input","Reads_lowqual", "Reads_aligned_all_amplicons","Reads_aligned_WT", "Reads_aligned_HDR", "Reads_unaligned", "WT","correct_edit", "target_base_edit", "byproduct_all","byproduct_INDEL","byproduct_sub","byproduct_ambiguous"]
        col_vals = [ sample_ids, rii, r_lowqual, raaa, ra_wt, ra_hdr, r_unaligned, no_edit, correct, target_base_editing, byproduct_all, byproduct_indel, byproduct_sub, byproduct_ambig]

        out = pd.DataFrame(dict(zip(cols, col_vals)))
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)

        df_edits.append(out)

    edit_dict = dict(zip(sample_id, df_edits))

    return edit_dict

In [3]:
def MLE_merge(edit_dict, samps_to_merge, new_name):
    """ 
    combine replicates to generate best estimate of sensor editing
    only works for combinations of 3 replicates (could be modified)

    MODIFIED TO WORK FOR MULTIPLE REPLICATES
    """
    cols_to_add = ['Reads_in_input', 'Reads_lowqual',
       'Reads_aligned_all_amplicons', 'Reads_aligned_WT', 'Reads_aligned_HDR',
       'Reads_unaligned', 'WT', 'correct_edit', 'target_base_edit',
       'byproduct_all', 'byproduct_INDEL', 'byproduct_sub',
       'byproduct_ambiguous',]
    
    comb_holder = []
    for k in samps_to_merge:
        
        a = edit_dict[k[0]]
        comb = a[cols_to_add]

        for jj in k[1:]:
            print(jj)
            j_df = edit_dict[jj]
            comb+=j_df[cols_to_add]
        
        #a = edit_dict[k[0]]
        #b = edit_dict[k[1]]
        #c = edit_dict[k[2]]

        #check that guide indeces match
        #assert list(a['Guide_ID'])==list(b['Guide_ID'])==list(c['Guide_ID']), 'indexes dont match'

        #a2 = a[cols_to_add]
        #b2 = b[cols_to_add]
        #c2 = c[cols_to_add]

        #comb = a2+b2+c2

        comb['Guide_ID'] = a['Guide_ID']

        #and reorganize columns
        col_order = ['Guide_ID'] + cols_to_add
        out = comb[col_order]

        #and then calcualte percentages
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)

        comb_holder.append(out)

    out_dict = dict(zip(new_name, comb_holder))

    return out_dict

In [4]:
samp_puro = ['D25-251027_guide_split_BARCODES',
 'D25-251028_guide_split_BARCODES',
 'D25-251029_guide_split_BARCODES',
 'D25-251035_guide_split_BARCODES',
 'D25-251036_guide_split_BARCODES',
 'D25-251037_guide_split_BARCODES',
 'D25-251041_guide_split_BARCODES',
 'D25-251042_guide_split_BARCODES',
 'D25-251043_guide_split_BARCODES']

sample_id_puro = ["D0_preEPO_rep1",
    "D0_preEPO_rep2",
    "D0_preEPO_rep3",
    "D5_puro_rep1",
    "D5_puro_rep2",
    "D5_puro_rep3",
    "D15_puro_rep1",
    "D15_puro_rep2",
    "D15_puro_rep3",]


samp_noPuro = ['D25-251027_guide_split_BARCODES',
 'D25-251028_guide_split_BARCODES',
 'D25-251029_guide_split_BARCODES',
 'D25-251032_guide_split_BARCODES',
 'D25-251033_guide_split_BARCODES',
 'D25-251034_guide_split_BARCODES',
 'D25-251038_guide_split_BARCODES',
 'D25-251039_guide_split_BARCODES',
 'D25-251040_guide_split_BARCODES']

sample_id_noPuro = ["D0_preEPO_rep1",
    "D0_preEPO_rep2",
    "D0_preEPO_rep3",
    "D5_noPuro_rep1",
    "D5_noPuro_rep2",
    "D5_noPuro_rep3",
    "D15_noPuro_rep1",
    "D15_noPuro_rep2",
    "D15_noPuro_rep3",]


noPuro_samples = pd.DataFrame(dict(zip(['file_name', 'sample'], [samp_noPuro, sample_id_noPuro])))
puro_samples = pd.DataFrame(dict(zip(['file_name', 'sample'], [samp_puro, sample_id_puro])))

noPuro_samp_dict = dict(zip(samp_noPuro, sample_id_noPuro))
puro_samp_dict = dict(zip(samp_puro, sample_id_puro))

# noPuro (formerly ABE) quantification and MLE generation

In [5]:
samp = list(noPuro_samp_dict.keys())
sample_id = list(noPuro_samp_dict.values())
folder_path = '/home/hcevasco/data/DASIT/251212San/crispresso'
noPuro_edit_dict = crispresso_compiler(samp, sample_id, folder_path)

In [7]:
samps_to_merge = [["D0_preEPO_rep1", "D0_preEPO_rep2","D0_preEPO_rep3"], 
                  ["D5_noPuro_rep1", "D5_noPuro_rep2","D5_noPuro_rep3"],
                  ["D15_noPuro_rep1","D15_noPuro_rep2","D15_noPuro_rep3"],
                 ]

new_name = ['D0_preEPO_MLE', 'D5_noPuro_MLE', 'D15_noPuro_MLE']
out_MLE = MLE_merge(noPuro_edit_dict, samps_to_merge, new_name)

out_MLE['D0_preEPO_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D0_preEPO_MLE.csv', sep='\t', index=False)
out_MLE['D5_noPuro_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D5_noPuro_MLE.csv', sep='\t', index=False)
out_MLE['D15_noPuro_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D15_noPuro_MLE.csv', sep='\t', index=False)


D0_preEPO_rep2
D0_preEPO_rep3
D5_noPuro_rep2
D5_noPuro_rep3
D15_noPuro_rep2
D15_noPuro_rep3


# Puro (formerly CBE) quantification and MLE generation

In [8]:
samp = list(puro_samp_dict.keys())
sample_id = list(puro_samp_dict.values())
folder_path = '/home/hcevasco/data/DASIT/251212San/crispresso'
puro_edit_dict = crispresso_compiler(samp, sample_id, folder_path)

I think these next two blocks are irrelevant for DASIT analysis. They were for legacy screen data

In [ ]:
""" #and then run it for the legacy guides that were quantified later
samp = list(puro_samp_dict.keys())
sample_id = list(puro_samp_dict.values())
folder_path = '240807HemA_CBE/crispresso_legacy'
puro_edit_dict_legacy = crispresso_compiler(samp, sample_id, folder_path) """

In [ ]:
""" #and updating the results so that the legacy guides have editing quantified
sample_id = list(puro_samp_dict.values())

new_dfs = []
for sample_considered in sample_id:
    a = puro_edit_dict_legacy[sample_considered]

    b = puro_edit_dict[sample_considered]

    b = b[~b['Guide_ID'].isin(list(a['Guide_ID']))]

    new = pd.concat((b,a)).sort_values(by='Guide_ID')
    new_dfs.append(new)


puro_edit_dict_new = dict(zip(sample_id, new_dfs)) """

In [9]:
samps_to_merge = [["D0_preEPO_rep1", "D0_preEPO_rep2","D0_preEPO_rep3"], 
                  ["D5_puro_rep1","D5_puro_rep2","D5_puro_rep3"],
                  ["D15_puro_rep1","D15_puro_rep2","D15_puro_rep3"]]

new_name = ['D0_preEPO_MLE', 'D5_puro_MLE', 'D15_puro_MLE']
out_MLE_puro = MLE_merge(puro_edit_dict, samps_to_merge, new_name)

#already have this MLE from the no puro set since it's the same data at day 0
#out_MLE['D0_preEPO_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D0_preEPO_MLE.csv', sep='\t', index=False)

out_MLE_puro['D5_puro_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D5_noPuro_MLE.csv', sep='\t', index=False)
out_MLE_puro['D15_puro_MLE'].to_csv('/net/bmc-lab2/data/lab/sanchezrivera/hcevasco/DASIT/251212San/D15_noPuro_MLE.csv', sep='\t', index=False)


D0_preEPO_rep2
D0_preEPO_rep3
D5_puro_rep2
D5_puro_rep3
D15_puro_rep2
D15_puro_rep3
